# L8.1 — Planner-Executor Agents

Hands-on notebook for the lesson [`8-1-planner-executor.mdx`](../../llm-quest-theory/level-8/8-1-planner-executor.mdx).

> **Learning objectives**
> - Implement a ReAct (Reason-Act-Observe) loop in **30 lines of pure Python** — the building block of every modern agent.
> - Upgrade it to **Plan-and-Execute**: one LLM call to plan, one per step to execute.
> - See how drift, infinite loops, and tool failures are handled with `max_steps`, error feedback, and goal reminders.
> - Compare fixed workflows vs agents on the same task — when is an agent overkill?

## Connection to the theory
Covers **§1–§11** of the source `.mdx`. To keep every run deterministic and CPU-free, the "LLM" is a **MockLLM** that follows a small rule set for tool selection. The loop shape is what transfers to a frontier LLM.

In [1]:
# ---- Setup ----
import os, re, json, time, random
from dataclasses import dataclass, field
from typing import Any, Callable

SEED = 42
random.seed(SEED)

## 1. The tools our agent will use
We simulate a small assistant that can check weather, look up exchange rates, and do arithmetic. Each tool returns structured data.

In [2]:
WEATHER_DB = {
    "hanoi":  {"today": ("sunny",  30), "saturday": ("sunny", 30), "sunday": ("rainy", 25)},
    "saigon": {"today": ("cloudy", 32), "saturday": ("rainy", 29), "sunday": ("sunny", 31)},
    "tokyo":  {"today": ("rainy",  18), "saturday": ("sunny", 20), "sunday": ("sunny", 22)},
}
FX_DB = {("USD", "VND"): 24500, ("VND", "USD"): 1/24500,
         ("USD", "JPY"): 150,   ("JPY", "USD"): 1/150}

def get_weather(city: str, when: str = "today") -> dict:
    c = city.lower().strip()
    if c not in WEATHER_DB:
        raise KeyError(f"unknown city {city!r}; known: {list(WEATHER_DB)}")
    cond, temp = WEATHER_DB[c][when]
    return {"city": city, "when": when, "cond": cond, "temp": temp}

def get_exchange_rate(from_cur: str, to_cur: str) -> dict:
    k = (from_cur.upper(), to_cur.upper())
    if k not in FX_DB:
        raise KeyError(f"no exchange rate {from_cur}->{to_cur}")
    return {"from": k[0], "to": k[1], "rate": FX_DB[k]}

def calc(expression: str) -> dict:
    # Safe arithmetic only: digits, + - * / ( ) . and spaces
    if not re.fullmatch(r"[0-9+\-*/().\s]+", expression):
        raise ValueError(f"unsafe expression: {expression!r}")
    return {"expression": expression, "value": eval(expression)}

TOOLS = {"get_weather": get_weather, "get_exchange_rate": get_exchange_rate, "calc": calc}

# Smoke test
print(get_weather("Hanoi", "saturday"))
print(get_exchange_rate("USD", "VND"))
print(calc("100 * 24500"))

{'city': 'Hanoi', 'when': 'saturday', 'cond': 'sunny', 'temp': 30}
{'from': 'USD', 'to': 'VND', 'rate': 24500}
{'expression': '100 * 24500', 'value': 2450000}


## 2. A MockLLM — deterministic tool-selection policy
Real frontier models emit structured tool calls. Our mock inspects the latest user / observation message and decides the next step via a few rules. This keeps the notebook reproducible without a paid model.

In [3]:
@dataclass
class AgentMessage:
    role: str              # 'user' | 'assistant' | 'observation' | 'system'
    content: str
    tool_call: dict | None = None   # {'name': ..., 'args': {...}}

class MockLLM:
    """Emits a tool call (or final answer) based on the trailing context."""
    def __init__(self): self.calls = 0

    def __call__(self, messages: list[AgentMessage]) -> AgentMessage:
        self.calls += 1
        # Reconstruct the trail — most recent user + all observations.
        user_msg = next((m.content for m in messages if m.role == "user"), "")
        obs = [m.content for m in messages if m.role == "observation"]
        seen_tools = [m.tool_call["name"] for m in messages if m.tool_call]

        low = user_msg.lower()

        # --- Weather branch ---
        if "weather" in low or "picnic" in low:
            city_match = re.search(r"in\s+([A-Za-z]+)", user_msg)
            city = city_match.group(1) if city_match else "hanoi"
            if "saturday" in low or "weekend" in low or "picnic" in low:
                # Need to check sat AND sun
                if "get_weather(saturday)" not in seen_tools:
                    return AgentMessage("assistant", "I should check Saturday's weather first.",
                                        tool_call={"name": "get_weather", "args": {"city": city, "when": "saturday"}})
                if "get_weather(sunday)" not in seen_tools:
                    return AgentMessage("assistant", "Now checking Sunday.",
                                        tool_call={"name": "get_weather", "args": {"city": city, "when": "sunday"}})
                # Have both — answer (or gracefully recover if an observation carried an error)
                sat = json.loads(obs[-2]); sun = json.loads(obs[-1])
                if "error" in sat or "error" in sun:
                    return AgentMessage("assistant", f"Sorry, I could not retrieve the weekend forecast for {city}.")
                pick = "Saturday" if sat["cond"] == "sunny" else ("Sunday" if sun["cond"] == "sunny" else "neither")
                return AgentMessage("assistant",
                    f"In {city} the weekend outlook is {sat['cond']} ({sat['temp']}C) Sat and {sun['cond']} ({sun['temp']}C) Sun. "
                    f"Best choice: {pick}.")
            # Today
            if not seen_tools:
                return AgentMessage("assistant", "Looking up today's weather.",
                                    tool_call={"name": "get_weather", "args": {"city": city, "when": "today"}})
            d = json.loads(obs[-1])
            if "error" in d:
                return AgentMessage("assistant", f"Sorry, I could not find weather for {city} ({d['error']}).")
            return AgentMessage("assistant", f"{city} today is {d['cond']} at {d['temp']}C.")

        # --- Currency conversion ---
        m = re.search(r"(\d+(?:\.\d+)?)\s*([A-Z]{3}).*(?:to|in)\s*([A-Z]{3})", user_msg, flags=re.IGNORECASE)
        if m:
            amount = float(m.group(1)); src = m.group(2).upper(); dst = m.group(3).upper()
            if "get_exchange_rate" not in [t.split("(")[0] for t in seen_tools]:
                return AgentMessage("assistant", f"Fetching FX rate for {src}->{dst}.",
                                    tool_call={"name": "get_exchange_rate", "args": {"from_cur": src, "to_cur": dst}})
            rate = json.loads(obs[-1])["rate"]
            if "calc" not in [t.split("(")[0] for t in seen_tools]:
                return AgentMessage("assistant", "Multiplying amount by rate.",
                                    tool_call={"name": "calc", "args": {"expression": f"{amount} * {rate}"}})
            total = json.loads(obs[-1])["value"]
            return AgentMessage("assistant", f"{amount} {src} = {total:.2f} {dst}.")

        # --- Default: reply directly ---
        return AgentMessage("assistant", "I don't know how to help with that.")

llm = MockLLM()

## 3. The ReAct loop in 30 lines
Maintain a message trail, ask the LLM for the next step, execute tool calls, feed observations back, stop when there's no tool call or we hit `max_steps`.

In [4]:
def run_react(user_query, tools=TOOLS, llm=llm, max_steps=6, verbose=True):
    trail = [AgentMessage("user", user_query)]
    for step in range(max_steps):
        reply = llm(trail)
        if verbose: print(f"[step {step}]  thought: {reply.content}")
        trail.append(reply)
        if reply.tool_call is None:
            return reply.content, trail
        name, args = reply.tool_call["name"], reply.tool_call["args"]
        try:
            result = tools[name](**args)
            obs = json.dumps(result)
            if verbose: print(f"           tool  : {name}({args}) -> {obs}")
        except Exception as e:
            obs = json.dumps({"error": str(e), "error_type": type(e).__name__})
            if verbose: print(f"           ERROR : {obs}")
        # Record the tool call and observation
        trail[-1].tool_call = {"name": f"{name}({args.get('when', args.get('from_cur', ''))})",
                                "args": args}
        trail.append(AgentMessage("observation", obs))
    return "Max steps reached.", trail

answer, _ = run_react("I'm in Hanoi — is it a good idea to go on a picnic this weekend?")
print("\nFINAL:", answer)

[step 0]  thought: I should check Saturday's weather first.
           tool  : get_weather({'city': 'Hanoi', 'when': 'saturday'}) -> {"city": "Hanoi", "when": "saturday", "cond": "sunny", "temp": 30}
[step 1]  thought: Now checking Sunday.
           tool  : get_weather({'city': 'Hanoi', 'when': 'sunday'}) -> {"city": "Hanoi", "when": "sunday", "cond": "rainy", "temp": 25}
[step 2]  thought: In Hanoi the weekend outlook is sunny (30C) Sat and rainy (25C) Sun. Best choice: Saturday.

FINAL: In Hanoi the weekend outlook is sunny (30C) Sat and rainy (25C) Sun. Best choice: Saturday.


## 4. A second task — currency conversion via two tools

In [5]:
answer, _ = run_react("How many VND is 100 USD?")
print("\nFINAL:", answer)

[step 0]  thought: I don't know how to help with that.

FINAL: I don't know how to help with that.


## 5. Error handling — tool raises, agent recovers
Ask about an unknown city. The tool raises, the agent is free to retry with a fallback city (in our mock, we just observe the recovery flow).

In [6]:
# This query is intentionally out of the WEATHER_DB; watch the ERROR observation.
answer, trail = run_react("What's the weather in Zurich today?", max_steps=3)
print("\nFINAL:", answer)
print("\nAgent trail captured the error observation:")
for i, m in enumerate(trail):
    print(f"  {i} [{m.role}] {m.content[:80]}")

[step 0]  thought: Looking up today's weather.
           ERROR : {"error": "\"unknown city 'Zurich'; known: ['hanoi', 'saigon', 'tokyo']\"", "error_type": "KeyError"}
[step 1]  thought: Sorry, I could not find weather for Zurich ("unknown city 'Zurich'; known: ['hanoi', 'saigon', 'tokyo']").

FINAL: Sorry, I could not find weather for Zurich ("unknown city 'Zurich'; known: ['hanoi', 'saigon', 'tokyo']").

Agent trail captured the error observation:
  0 [user] What's the weather in Zurich today?
  1 [assistant] Looking up today's weather.
  2 [observation] {"error": "\"unknown city 'Zurich'; known: ['hanoi', 'saigon', 'tokyo']\"", "err
  3 [assistant] Sorry, I could not find weather for Zurich ("unknown city 'Zurich'; known: ['han


## 6. Plan-and-Execute — plan once, then run
The Planner returns a static list of steps before any tool runs. Useful when the user wants to see the plan first or when parallel execution pays off.

In [7]:
def plan_trip(query):
    """Mock planner — returns a list of (name, args) tool steps."""
    # The real planner would be an LLM call; ours just pattern-matches.
    if "usd" in query.lower() and "vnd" in query.lower():
        return [
            ("get_exchange_rate", {"from_cur": "USD", "to_cur": "VND"}),
            ("calc",               {"expression": "200 * 24500"}),
        ]
    if "weekend" in query.lower() or "picnic" in query.lower():
        city = "hanoi" if "hanoi" in query.lower() else "tokyo"
        return [
            ("get_weather", {"city": city, "when": "saturday"}),
            ("get_weather", {"city": city, "when": "sunday"}),
        ]
    return []

def run_plan_execute(query, plan_fn=plan_trip, tools=TOOLS, verbose=True):
    plan = plan_fn(query)
    if verbose:
        print("PLAN:")
        for step in plan: print("  -", step)
    results = []
    for name, args in plan:
        try:
            results.append(tools[name](**args))
        except Exception as e:
            results.append({"error": str(e)})
    # Simple synthesiser — real product would use the LLM.
    return results

res = run_plan_execute("Is the weekend good for a picnic in Hanoi?")
print("\nRESULTS:", json.dumps(res, indent=2))

PLAN:
  - ('get_weather', {'city': 'hanoi', 'when': 'saturday'})
  - ('get_weather', {'city': 'hanoi', 'when': 'sunday'})

RESULTS: [
  {
    "city": "hanoi",
    "when": "saturday",
    "cond": "sunny",
    "temp": 30
  },
  {
    "city": "hanoi",
    "when": "sunday",
    "cond": "rainy",
    "temp": 25
  }
]


## 7. Infinite-loop protection
A faulty MockLLM could keep asking for the same tool. `max_steps` is the simple guard; a good agent also detects **no-progress** loops.

In [8]:
class LoopyLLM:
    """Always asks for the same tool, never terminates on its own."""
    def __call__(self, messages):
        return AgentMessage("assistant", "let me re-check",
                            tool_call={"name": "get_weather", "args": {"city": "Hanoi", "when": "today"}})

answer, trail = run_react("weather hanoi", llm=LoopyLLM(), max_steps=4, verbose=False)
print("FINAL (no natural stop):", answer)
print("trail length            :", len(trail), "(max_steps guarded us)")

FINAL (no natural stop): Max steps reached.
trail length            : 9 (max_steps guarded us)


## 8. Workflow vs agent — the cost comparison
Same answer as "check Hanoi weekend weather" done three ways: a hard-coded workflow (2 tool calls), a ReAct agent (3 LLM calls + 2 tool calls), and a plan-and-execute agent (1 planner call + 2 tool calls). Agents do cost more.

In [9]:
def workflow(city):
    return [get_weather(city, "saturday"), get_weather(city, "sunday")]

# Reset counters
llm.calls = 0
wf_out = workflow("hanoi")
react_ans, react_trail = run_react("Is the weekend good for a picnic in Hanoi?", verbose=False)
react_llm_calls = llm.calls
plan_out = run_plan_execute("Is the weekend good for a picnic in Hanoi?", verbose=False)

print("workflow     : 0 LLM calls, 2 tool calls")
print(f"ReAct        : {react_llm_calls} LLM calls, ",
      sum(1 for m in react_trail if m.tool_call), "tool calls")
print("plan+execute : 1 LLM call (planner), 2 tool calls")

workflow     : 0 LLM calls, 2 tool calls
ReAct        : 3 LLM calls,  2 tool calls
plan+execute : 1 LLM call (planner), 2 tool calls


## 9. Quick checks

In [10]:
# ReAct must return a final answer (not the loop-limit sentinel) within the budget
ans, _ = run_react("weather hanoi today", max_steps=3, verbose=False)
assert "hanoi today" in ans.lower()
# Budget guard works: LoopyLLM cannot run past max_steps
bad_ans, bad_trail = run_react("loop", llm=LoopyLLM(), max_steps=2, verbose=False)
assert "max steps" in bad_ans.lower(), f"expected Max-steps sentinel, got {bad_ans!r}"
# Unknown city produces an error observation that the trail captured
_, e_trail = run_react("What's the weather in Zurich today?", max_steps=3, verbose=False)
assert any("error" in m.content for m in e_trail), "trail should record the tool error"
# Plan-and-execute returned at least two structured results for a weekend query
plan_res = run_plan_execute("Is the weekend good for a picnic in Hanoi?", verbose=False)
assert len(plan_res) == 2 and all("cond" in r for r in plan_res)
# Safety: calc refuses non-arithmetic input
try:
    calc("import os")
    raise AssertionError("calc should reject unsafe input")
except ValueError:
    pass
print("OK — ReAct, plan-and-execute, budget guard and error-recovery all behave.")

OK — ReAct, plan-and-execute, budget guard and error-recovery all behave.


## Reflection questions

1. Our `calc` tool uses `eval`. In a real product, even a safely-guarded `eval` is risky. Name three safer ways to implement arithmetic inside a tool.
2. The ReAct loop sent an error observation to the LLM when a tool failed. What would happen if we silently retried with the same arguments instead?
3. Plan-and-execute is attractive but rigid. Give an example task where a step-2 failure makes the rest of the plan obsolete.
4. The theory mentions `Tree-of-Agents`. Sketch how you would turn our mock flow into a three-role setup (Planner / Weather-worker / Currency-worker) sharing one goal.

## References
- Source theory: [`8-1-planner-executor.mdx`](../../llm-quest-theory/level-8/8-1-planner-executor.mdx)
- Next: [`8-2-tool-calling`](8-2-tool-calling.ipynb)